In [1]:
import watermark
pkg_versions = watermark.watermark(
    packages="requests,pandas,tqdm")
print(pkg_versions)

requests: 2.32.5
pandas  : 1.5.3
tqdm    : 4.67.1



In [2]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://apis.data.go.kr"

# 행정안전부_행정표준코드_법정동코드

In [3]:
URL = f"{BASE_URL}/1741000/StanReginCd/getStanReginCdList"
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1000,
    "pageNo": 1,
    "flag":"Y",
    "locatadd_nm":"부산광역시",
    "type":"json"
}
res = requests.get(URL, params= params)
data = res.json()
station_df = pd.DataFrame(data["StanReginCd"][1]["row"])
station_df["signguCode"] = (
    station_df["sido_cd"].astype(str).str.zfill(2)
    + station_df["sgg_cd"].astype(str).str.zfill(3)
)
busan_df = station_df[
    station_df["locallow_nm"].str.contains(
        r".*(?:구|군)$", na=False
    )
].reset_index(drop=True)

In [4]:
busan_df

,region_cd,sido_cd,sgg_cd,umd_cd,ri_cd,locatjumin_cd,locatjijuk_cd,locatadd_nm,locat_order,locat_rm,locathigh_cd,locallow_nm,adpt_de,signguCode
0,2611000000,26,110,000,00,2611000000,2611000000,부산광역시 중구,1,,2600000000,중구,,26110
1,2614000000,26,140,000,00,2614000000,2614000000,부산광역시 서구,2,,2600000000,서구,,26140
2,2617000000,26,170,000,00,2617000000,2617000000,부산광역시 동구,3,,2600000000,동구,,26170
3,2620000000,26,200,000,00,2620000000,2620000000,부산광역시 영도구,4,,2600000000,영도구,,26200
4,2623000000,26,230,000,00,2623000000,2623000000,부산광역시 부산진구,5,,2600000000,부산진구,,26230
5,2626000000,26,260,000,00,2626000000,2626000000,부산광역시 동래구,6,,2600000000,동래구,,26260
6,2629000000,26,290,000,00,2629000000,2629000000,부산광역시 남구,7,,2600000000,남구,,26290
7,2632000000,26,320,000,00,2632000000,2632000000,부산광역시 북구,8,,2600000000,북구,,26320
8,2635000000,26,350,000,00,2635000000,2635000000,부산광역시 해운대구,9,,2600000000,해운대구,,26350
9,2638000000,26,380,000,00,2638000000,2638000000,부산광역시 사하구,10,,2600000000,사하구,,26380


In [5]:
busan_codes = busan_df["signguCode"].unique()

# 부산광역시_부산맛집정보 서비스

In [6]:
URL = f"{BASE_URL}/6260000/FoodService/getFoodKr"

In [7]:
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1,
    "pageNo": 1,
    "resultType":"json"
}

In [22]:
res = requests.get(URL, params= params)
data = res.json()
total_count = data["getFoodKr"]["totalCount"]
total_count

437

In [23]:
num_of_rows = 1000
params.update({"numOfRows":num_of_rows})

dfs = list()
for page in tqdm(range(1, total_count//num_of_rows + 2)):
    params.update({"pageNo":page})
    res = requests.get(URL, params= params)
    data = res.json()
    df = pd.DataFrame(data["getFoodKr"]["item"])
    dfs.append(df)

100%|██████████| 1/1 [00:00<00:00,  3.01it/s]


In [34]:
df = pd.concat(dfs,ignore_index=True)
df.head()

,UC_SEQ,MAIN_TITLE,GUGUN_NM,LAT,LNG,PLACE,TITLE,SUBTITLE,ADDR1,ADDR2,CNTCT_TEL,HOMEPAGE_URL,USAGE_DAY_WEEK_AND_TIME,RPRSNTV_MENU,MAIN_IMG_NORMAL,MAIN_IMG_THUMB,ITEMCNTNTS
0,70,만드리곤드레밥,강서구,35.177387,128.95245,만드리곤드레밥,만드리곤드레밥,,강서구 공항앞길 85번길 13,,051-941-3669,,10:00-20:00\n(19:50 라스트오더),"돌솥곤드레정식, 단호박오리훈제",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"곤드레밥에는 일반적으로 건조 곤드레나물이 사용되는데,\n이곳은 생 곤드레나물을 사용..."
1,77,민물가든,강서구,35.160550,128.89468,민물가든,민물가든,민물가든,강서구 둔치중앙길5(봉림동),,051-971-8428,https://blog.naver.com/rladba1,24.03.12 ~ 24. 12.31 휴업중\n11:00a.m. ~ 21:00p.m...,"묵은지붕어조림, 붕어찜",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"30년간 운영해온 생선찜전문점으로, 전통방식인 나무통을 사용하여 조리하는 것이 특징..."
2,94,가야할매밀면,연제구,35.185196,129.07988,가야할매밀면,가야할매밀면,가야할매밀면,부산 연제구 월드컵대로 145번길 32\n,,051-865-8017,,11:00-20:00,"물 밀면, 비빔밀면\n",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"유명한 밀면전문점으로, 밀가루에 옥수수 전분을 섞어, 다른 밀면전문점들보다 더 탄력..."
3,95,국제밀면본점,연제구,35.196890,129.07785,국제밀면본점,국제밀면본점,,연제구 중앙대로1235번길 23-6,,051-501-5507,,10:00-20:00 (라스트오더 19:30)\n5~8월 10:00-21:00 (라...,"물밀면 ￦9,000\n비빔밀면 ￦9,000",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,밀면전문점 중에서도 특히 맛으로 소문난 이곳은\n소 사골만을 사용한 육수 등 독창적...
4,102,할매가야밀면,중구,35.098934,129.03122,할매가야밀면,할매가야밀면,,중구 광복로 56-14,,051-246-3314,,10:30-21:30,"밀면, 비빔밀면",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"깔끔한 맛의 육수가 인상적인 40년 전통의 밀면 전문점으로, \n식사시간에는 항상 ..."


In [35]:
df.shape

(437, 17)

In [38]:
merged_df = pd.merge(busan_df[["locallow_nm","signguCode"]], df, how="left", left_on="locallow_nm", right_on="GUGUN_NM")
merged_df.head()

,locallow_nm,signguCode,UC_SEQ,MAIN_TITLE,GUGUN_NM,LAT,LNG,PLACE,TITLE,SUBTITLE,ADDR1,ADDR2,CNTCT_TEL,HOMEPAGE_URL,USAGE_DAY_WEEK_AND_TIME,RPRSNTV_MENU,MAIN_IMG_NORMAL,MAIN_IMG_THUMB,ITEMCNTNTS
0,중구,26110,102,할매가야밀면,중구,35.098934,129.03122,할매가야밀면,할매가야밀면,,중구 광복로 56-14,,051-246-3314,,10:30-21:30,"밀면, 비빔밀면",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"깔끔한 맛의 육수가 인상적인 40년 전통의 밀면 전문점으로, \n식사시간에는 항상 ..."
1,중구,26110,103,거인통닭,중구,35.102345,129.02612,거인통닭,거인통닭,,중구 중구로47번길34,,051-246-6079,,11:30-20:00\n13:30-14:30 브레이크 타임,후라이드치킨,https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"3대에 걸쳐 약 60여 년을 운영 중인 유명한 식당으로, 프라이드치킨 단 하나의 메..."
2,중구,26110,118,부산꼼장어맛집 성일집,중구,35.099426,129.03749,부산꼼장어맛집 성일집,부산꼼장어맛집 성일집,,중구 대교로 103,,0507-1345-5890,http://www.ggom.co.kr/,11:00-23:00\n(22:00 라스트오더),"소금구이, 양념구이",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,1950년부터 3대에 걸쳐 74년째 운영 중인 곳으로 가장 유명한 곰장어전문점 중 ...
3,중구,26110,1137,원산면옥,중구,35.098976,129.03105,원산면옥 1953,원산면옥 1953,,중구 광복로 56-8,,051-245-2310,http://wonsannoodle.fordining.kr/,11:00-21:30,"평양냉면 ￦14,000\n함흥냉면 ￦14,000",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,60여 년의 전통을 가진 냉면전문점으로 3대째 운영하고 있는 무척 유명한 곳이다.\...
4,중구,26110,1239,부산족발,중구,35.099560,129.02664,부산족발,부산족발,,중구 광복로 13-1,,0507-1412-5359,,10:00-24:00,"족발, 냉채족발",https://www.visitbusan.net/uploadImgs/files/cn...,https://www.visitbusan.net/uploadImgs/files/cn...,"부평동 돼지족발골목에 위치한 족발전문점으로, TV 프로그램으로도 자주 소개되는 유명..."


In [39]:
merged_df.to_csv("부산광역시_부산맛집정보 서비스-국문.csv", index=False)